# Opening the black box that is machine learning models

# MSc Computer Science - Artificial Intelligence

ID: 23200322

This notebook holds the experiments carried out as part of the thesis.

## Experiment 1:
1. Run 4 models on the ETSAP-TIAM dataset from [1], subsetted by scenarios, targets, and x variable subsets.
2. Collect performance metrics from each model.
3. Generate SHAP plots for each model on each target/scenario/x-varaible subset combination.


## Experiment 2: 
1. Run a parameter sweep on novel symbolic regression algorithm 'SBGP', altering the value of init_prop.
2. Collect performance statistics and benchmark against PySR.


Note: The novel symbolic regression algorithm use in this thesis was originally called 'Magpie', but towards the end of this thesis, it was renamed to 'SBGP' due to naming conflicts with other symbolic regression algorithms. Magpie may still appear in some code used/plots generated. 

[1] J. McDermott, J. Glynn, I. Morrow, and E. Panos, “Symbolic Regression for Modelling Decarbonisation Pathways in the Global Energy-Economy-Climate System,” in Proceedings of the Genetic and Evolutionary Computation Conference Companion, in GECCO ’25 Companion. New York, NY, USA: Association for Computing Machinery, Aug. 2025, pp. 871–874. doi: 10.1145/3712255.3726704.


In [ ]:
from collections import defaultdict
from datetime import datetime
import itertools
import Magpie
from Magpie import MagpieRegressor
import math
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
pd.set_option("display.max_colwidth", None)
from pysr import PySRRegressor
import re
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.neural_network import MLPRegressor
import sympy as sp
import time
import warnings
warnings.filterwarnings("ignore")

                                       
run_date = datetime.now().strftime('%Y%m%d')
matplotlib.rcParams['pdf.fonttype'] = 42 # helps produce ACM-compliant figures
matplotlib.rcParams['ps.fonttype'] = 42
Xy = pd.read_csv(r"C:\Users\Sophie\OneDrive - National University of Ireland, Galway\Y1S2\Thesis\GECCO Code\Outputs\data_Xy_clim_sens_filter.csv", index_col=0)


In [ ]:
"""
features = [
 'GDP',
 'Pop',
 'Other_ESD_Drivers',
 'SDR',
 'Elast_ESD_Driver',
 'Elast_ESD_Price',
 'CO2_Storage_Poten',
 'Wind_Poten',
 'Solar_Poten',
 'Biomass_Poten',
 'Oil_Gas_Poten',
 'Solar_PV_Inv_Cost',
 'Wind_Inv_Cost',
 'Bioenergy_CCS_Inv_Cost',
 'Other_Tech_Cost',
 'Forcing',
 'Land_Sinks',
 'Clim_Sens',
 'Year']
"""

### xvar_subsets -
 There are two sets of X-variable subsets: 'small' and 'all'. The reason for this is that, in [1], some variables are considered to be not super impactful on the value of the prediction. 

### Scenarios -
 Temperature limits and delays are the two factors that make up each scenario

### Targets - 
  Targets are the values to be predicted

In [ ]:

xvar_subsets = {
    'small': ['SDR', 'Clim_Sens', 'Pop', 'GDP'],
    'all': ['GDP', 'Pop',
       'Other_ESD_Drivers', 'SDR', 'Elast_ESD_Driver', 'Elast_ESD_Price',
       'CO2_Storage_Poten', 'Wind_Poten', 'Solar_Poten', 'Biomass_Poten',
       'Oil_Gas_Poten', 'Solar_PV_Inv_Cost', 'Wind_Inv_Cost',
       'Bioenergy_CCS_Inv_Cost', 'Other_Tech_Cost', 'Forcing', 'Land_Sinks',
       'Clim_Sens']
}

scenarios = ['BASE_SSP2','1p5c_OS_SSP2', '2C_SSP2', '2C_SSP2_DA30']

targets = ['GCost', 'GSupply', 'CO2eq', 'GConsumption' ] 

### Experiment 1:
In this experiment the code loops through each subset combination of scenario/target/x-variable subset. 

4 models are ran on each subset combination : PySR, Magpie/SBGP/RFR and MLP Regressor

Performance statistics are collected from all models and for the Symbolic Regression models, the top 5 equations from each run are collected

SHAP plots are generated for each model/scenario/target/x-variable subset combination

In [ ]:
base_dir = Path(run_date) / "Shap_outputs"
base_dir.mkdir(parents=True, exist_ok=True)

def run_everything(Xy):
####################################################################################
# Beginning the sweep of comparing each target x scenario x subset x model. 
# Each model is trained on the subset of parameters and model metrics are generated.
# Shap_results stores the model metrics. SHAP is also initiated within this cell, and 
# 6 different graph types are generated for each model on each parameter combination
#####################################################################################
    shap_results = []  
    pysr_equation_rows = []
    magpie_equation_rows = []                                                                          
    for target in targets:                                                                  
        for scenario in scenarios:                                                          
            if scenario == 'ALL':
                Xy_sub = Xy
            else: 
                Xy_sub = Xy[Xy['Scenario'] == scenario]
            for xvar_subset_code in xvar_subsets:
                # for the 'ALL' scenario, run only if we have _inc_enc
                if scenario == 'ALL' and (not "_inc_enc" in xvar_subset_code): 
                    continue
                # for the non-ALL scenarios, run only if we DO NOT have _inc_enc
                if scenario != 'ALL' and ("_inc_enc" in xvar_subset_code): 
                    continue

                xvar_subset = xvar_subsets[xvar_subset_code]
                X = Xy_sub[xvar_subset]
                y = Xy_sub[target]
                print(f'PARAMETERS: Scenario {scenario}, xvar code {xvar_subset_code} xvars {xvar_subset}, target {target}\n')
                print("")

                # Model instances
                models = {
                    'Magpie Regressor': MagpieRegressor(
                        maxevals=10000,                     # 10,000
                        init_prop=.25                  
                    ),
                    'PySRRegressor': PySRRegressor(
                        niterations=20,
                        binary_operators=["+", "*", "-", "/"],
                        unary_operators=["cos", "exp", "sin", "log"],
                        max_evals = 10000,                                          # this is to create equivalency between mapie and Pysr with the amount of equations they evaluate. 
                        random_state=42, 
                        verbosity=0,
                        deterministic=True,
                        parallelism='serial'
                    ),
                    'RandomForestRegressor': RandomForestRegressor(
                        n_estimators=500,                                 
                        max_depth=3,
                    ),
                    'MLPRegressor': MLPRegressor(
                        hidden_layer_sizes=(10,), 
                        max_iter=500,                                   
                        early_stopping=False
                    )
                }
                
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

                for model_name, model in models.items():                                            # Iterating over key-value pairs
                    print(f"TRAINING: {model_name}...\n")
                    train_start = time.perf_counter()                                               # Timing how long the model takes to train
                    model.fit(X_train, y_train)                                                     # Training model
                    train_time = time.perf_counter() - train_start                
                    y_pred_train = model.predict(X_train)
                    pred_start = time.perf_counter()
                    y_pred_test = model.predict(X_test)
                    prediction_time = time.perf_counter() - pred_start                              # Timing how long the model takes to predict
                    
                                                                                 
                    mse_train = mean_squared_error(y_train, y_pred_train)                           # Collecting performance statistics
                    mse_test = mean_squared_error(y_test, y_pred_test)
                    rmse_train = np.sqrt(mse_train)
                    rmse_test = np.sqrt(mse_test)
                    mae_train = mean_absolute_error(y_train, y_pred_train)
                    mae_test = mean_absolute_error(y_test, y_pred_test)
                    r2_train = r2_score(y_train, y_pred_train)
                    r2_test = r2_score(y_test, y_pred_test)
                                                                                                                
                    top_5_pysr = pd.DataFrame(columns=["complexity", "loss", "score", "equation"])              #Saving top 5 equations for each SR model
                    top_5_magpie = pd.DataFrame(columns=["size", "loss", "loss_validation", "equation"])        
                                                                                            
                    if model_name == 'PySRRegressor':                                                                     
                        try:
                            top_5_pysr = model.equations_.sort_values(by='loss').head(5)                                                            
                            best_equation = str(model.sympy())
                        except Exception as e:
                            best_equation = "Equation not available"
                                 
                        tagged = top_5_pysr[["complexity", "loss", "score", "equation"]].copy()
                        tagged["scenario"] = scenario
                        tagged["target"] = target
                        tagged["xvar_subset"] = xvar_subset_code
                        pysr_equation_rows.append(tagged)

                                                                                                        
                    elif model_name == 'Magpie Regressor':                                                                                                      
                        try:
                            top_5_magpie = model.equations_.sort_values(by='loss_validation').head(5)
                            best_row = model.equation_.iloc[0]
                            best_equation = str(best_row['latex'])
                            
                            
                        except Exception as e:
                            best_equation = "Equation not available"                                                                                                                                                                                     
                    
                        tagged = top_5_magpie[["size", "loss", "loss_validation", "equation", "latex"]].copy()
                        tagged["latex_stripped"] = tagged["latex"].str.replace(r'\\left|\\right', '', regex=True)
                        tagged["scenario"] = scenario
                        tagged["target"] = target
                        tagged["xvar_subset"] = xvar_subset_code
                        magpie_equation_rows.append(tagged)
                    
                    else:                                                                                              
                        if hasattr(model, 'loss_curve_'):
                            print(f"  Training curve (last 5 losses): {model.loss_curve_[-5:]}")
                        best_equation = f"{model_name} (no symbolic equation)"                                      

                    shap_results.append({
                     
                        'model_name': model_name,
                        'scenario': scenario,
                        'target': target,
                        'xvar_subset': xvar_subset_code,
                        'mse_train': mse_train,
                        'mse_test': mse_test,
                        'mae': mae_test,
                        'train_test_mse_ratio': mse_train/mse_test,
                        'r2_train': r2_train,
                        'r2_test': r2_test,
                        'train-test_r2_diff':r2_train - r2_test,
                        'best_equation': best_equation, 
                        'train_time': train_time,
                        'prediction_time':prediction_time,
                        'NRMSE': rmse_test / (y_test.max() - y_test.min())

                    })



            
                    #################################################
                    # SHAP is being initialized and implemented here
                    #################################################
                                    
                    background = X_train.iloc[np.random.choice(X_train.shape[0], 100, replace=False)]                    # Selecting background data (100-200 samples) for SHAP  

                    
                    explainer = shap.Explainer(model.predict, background)                                                # Initializing the explainer
                    print(f"EXPLAINER TYPE {type(explainer)}")
                    shap_values = explainer(X_test)                                                                      # Computing SHAP values using entire X_test dataset

                    plots = ['bar', 'scatter', 'heatmap', 'beeswarm', 'violin', 'waterfall']                             # Creating multiple different SHAP plot types
                    for plot in plots:
                        plt.figure(figsize=(10, 6))
                        if plot == 'bar':
                            shap.plots.bar(shap_values, max_display=10, show=False)
                        elif plot == 'scatter':
                            shap.plots.scatter(shap_values, show=False)
                        elif plot == 'heatmap':
                            shap.plots.heatmap(shap_values, instance_order=shap_values.sum(1), show=False)
                        elif plot == 'violin':
                            shap.plots.violin(shap_values, show=False)
                        elif plot == 'beeswarm':
                            shap.plots.beeswarm(shap_values, show=False)
                        elif plot == 'waterfall':
                            shap.plots.waterfall(shap_values[0], max_display=10, show=False)

                        fig = plt.gcf()

                        if plot == 'scatter':
                            for ax in fig.get_axes():
                                ax.set_xlabel(ax.get_xlabel(), rotation=45, ha='right')
                                ax.tick_params(axis='x', rotation=45)

                        output_dir = f"{base_dir}/{model_name}/{scenario}/{target}/{xvar_subset_code}"
                        os.makedirs(output_dir, exist_ok=True)
                        file_path = f"{output_dir}/{plot}.pdf"
                        file_path1 = f"{output_dir}/{plot}.png"
                        fig.savefig(file_path, bbox_inches='tight')
                        fig.savefig(file_path1, bbox_inches='tight')
                        plt.close(fig)
                        print(f"Saved plots for {model_name} to: {os.path.abspath(output_dir)}\n\n")
  
                        
    shap_df = pd.DataFrame(shap_results)
    pysr_equations_df = pd.concat(pysr_equation_rows, ignore_index=True) if pysr_equation_rows else pd.DataFrame(columns=["scenario","target","xvar_subset","complexity","loss","score","equation"])
    magpie_equations_df = pd.concat(magpie_equation_rows, ignore_index=True) if magpie_equation_rows else pd.DataFrame(columns=["scenario","target","xvar_subset","size","loss","loss_validation","equation", "latex", "latex_stripped"])

    os.makedirs(base_dir, exist_ok=True)
    shap_df.to_csv(f"{base_dir}/model_comparisson_results.csv", index=False)
    pysr_equations_df.to_csv(f"{base_dir}/top5_equations_pysr.csv", index=False)
    magpie_equations_df.to_csv(f"{base_dir}/top5_equations_magpie.csv", index=False)

    return shap_df, pysr_equations_df, magpie_equations_df

SHAP_DF, PYSR_EQUATIONS_DF, MAGPIE_EQUATIONS_DF = run_everything(Xy)


## Experiment 2:
This code carries out the parameter sweep for Magpie/SBGP. 

The value of init_props is varied and performance metrics are gathered on all scenario/target/x-variable subset combinations.

Each init_prop/scenario/target/x-variable subset combinations is ran x30 to allow for calculation of mean statistics

PySR is ran x30 also to allow for benchmark comparison


In [ ]:
max_evals = 10000                               
init_mults = [0.01, 0.05, 0.1, 0.3, 0.5]                                                             # Defining parameter sweep combinations     
magpie_params = [(max_evals, x) for x in init_mults]

param_sweep_results = []

for target in targets:                                                                               # Main Loop (Same as experiment 1)
    for scenario in scenarios:
        if scenario == 'ALL':
            Xy_sub = Xy
        else: 
            Xy_sub = Xy[Xy['Scenario'] == scenario]

        for xvar_subset_code in xvar_subsets:                                                                               
                                                                                                                            
            if scenario == 'ALL' and (not "_inc_enc" in xvar_subset_code): 
                continue
                                                                                                                            
            if scenario != 'ALL' and ("_inc_enc" in xvar_subset_code): 
                continue

            xvar_subset = xvar_subsets[xvar_subset_code]                                            # The x-variable subset currently being used is assigned the name 'xvar_subset'                                                             
            X = Xy_sub[xvar_subset]                                                                 # splitting the subset into its x and y components
            y = Xy_sub[target]

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            param_sweep_dir = Path(run_date) / "Param Sweep_outputs"
            param_sweep_dir.mkdir(parents=True, exist_ok=True)


            
            
        
                                                                                                    # Magpie Param Sweep
            for maxevals, init_prop in magpie_params:                                               # For each (max_evals, init_evals) combination (as defined above),                                                                                                                                                                                  # changed from 'initevals=init_evals' to 'init_prop = init_mult'
                print(f"Init Proportion = {init_prop}, Max Evals = {maxevals}")
                
                for i in range(30):
                    model = None  
                    try:
                        model_name = f'Magpie_max{maxevals}_init{init_prop}'
                        model = MagpieRegressor(
                            maxevals=maxevals,
                            init_prop=init_prop
                        )
                        
                        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Run {i}...")
                     
                        train_start = time.perf_counter()                                               
                        model.fit(X_train, y_train)  
                        train_time = time.perf_counter() - train_start                
                        y_pred_train = model.predict(X_train)
                        pred_start = time.perf_counter()
                        y_pred_test = model.predict(X_test)
                        prediction_time = time.perf_counter() - pred_start                              
                        
                                                                                    
                        mse_train = mean_squared_error(y_train, y_pred_train)
                        mse_test = mean_squared_error(y_test, y_pred_test)
                        rmse_train = np.sqrt(mse_train)
                        rmse_test = np.sqrt(mse_test)
                        mae_train = mean_absolute_error(y_train, y_pred_train)
                        mae_test = mean_absolute_error(y_test, y_pred_test)
                        r2_train = r2_score(y_train, y_pred_train)
                        r2_test = r2_score(y_test, y_pred_test)

                        
                        try:                                                                                    # Best equation for Magpie/SBGP
                            best_row = model.equations_.loc[model.equations_["loss_validation"].idxmin()] 
                            best_equation = str(best_row['equation'])
                            print(f"\nBest equation: {best_equation}")
                        except Exception as e:
                            print(f"Could not retrieve equations: {e}")
                            best_equation = "Equation not available"
                           
                        

                        param_sweep_results.append({                                                            # Collecting Stats
                            'run':i,
                            'model_name': 'Magpie',
                            'scenario': scenario,
                            'target': target,
                            'xvar_subset': xvar_subset_code,
                            'param1': maxevals,
                            'param2':init_prop,
                            'mse_train': mse_train,
                            'mse_test': mse_test,
                            'mae': mae_test,
                            'train_test_mse_ratio': mse_train/mse_test,
                            'r2_train': r2_train,
                            'r2_test': r2_test,
                            'train-test_r2_diff':r2_train - r2_test,
                            'best_equation': best_equation, 
                            'train_time': train_time,
                            'prediction_time':prediction_time,
                            'NRMSE': rmse_test / (y_test.max() - y_test.min())

                        })
                    
                    except Exception as e:
                        print(f"EXCEPTION in {model_name} run {i}: {e}")
                        import traceback
                        traceback.print_exc()


            model = PySRRegressor(                                                                      # PySR run
                max_evals= max_evals,
                parallelism='serial',
                deterministic=True
            )

            for i in range(30):
                model_name = f'PySR_niter{max_evals}'
                print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] TRAINING: {model_name} RUN: {i}...\n")

                try:
                    train_start = time.perf_counter()                                                   # Running Model
                    model.fit(X_train, y_train)  
                    train_time = time.perf_counter() - train_start                
                    y_pred_train = model.predict(X_train)
                    pred_start = time.perf_counter()
                    y_pred_test = model.predict(X_test)
                    prediction_time = time.perf_counter() - pred_start                              
                                                                            
                    mse_train = mean_squared_error(y_train, y_pred_train)                               # Collecting stats
                    mse_test = mean_squared_error(y_test, y_pred_test)
                    rmse_train = np.sqrt(mse_train)
                    rmse_test = np.sqrt(mse_test)
                    mae_train = mean_absolute_error(y_train, y_pred_train)
                    mae_test = mean_absolute_error(y_test, y_pred_test)
                    r2_train = r2_score(y_train, y_pred_train)
                    r2_test = r2_score(y_test, y_pred_test)

            
                    try:                                                                                # Collecting top equations
                        best_row = model.equations_.loc[model.equations_["loss"].idxmin()]
                        best_equation = str(best_row['equation'])
                        print(f"\nBest equation: {best_equation}")
                    except Exception as e:
                        print(f"Could not retrieve equations: {e}")
                        best_equation = "Equation not available"
                
                    param_sweep_results.append({                                                        # Storing results
                            'run':i,
                            'model_name': 'PySR',
                            'scenario': scenario,
                            'target': target,
                            'xvar_subset': xvar_subset_code,
                            'param1': max_evals,
                            'param2':None,
                            'mse_train': mse_train,
                            'mse_test': mse_test,
                            'mae': mae_test,
                            'train_test_mse_ratio': mse_train/mse_test,
                            'r2_train': r2_train,
                            'r2_test': r2_test,
                            'train-test_r2_diff':r2_train - r2_test,
                            'best_equation': best_equation, 
                            'train_time': train_time,
                            'prediction_time':prediction_time,
                            'NRMSE': rmse_test / (y_test.max() - y_test.min())

                        })
                    
                except Exception as e:
                    print(f"EXCEPTION in {model_name}: {str(e)}")
                    import traceback
                    print(traceback.format_exc())
                    continue

param_sweep_df = pd.DataFrame(param_sweep_results)
param_sweep_df.to_csv(f"Param Sweep_{run_date}/parameter_sweep_results.csv", index=False)


print(param_sweep_df.columns)
print(param_sweep_df.head())

The following cells contain code that was used to create plots based on the data generated in experiments 1 and 2

In [ ]:
#################################################
# The Heatmap — one subplot per model
#################################################

def get_text_colour(value, vmin, vmax, cmap):                                                                       # Please note the method 'get_text_colour' was generated using Claude
    """Return 'white' or 'black' depending on the luminance of the cell's color."""
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    rgba = cmap(norm(value))
    r, g, b = rgba[0], rgba[1], rgba[2]
    # Perceptual luminance (standard weights)
    luminance = 0.299 * r + 0.587 * g + 0.114 * b
    return 'white' if luminance < 0.5 else 'black'

def plot_heatmap(metric, feature_subset):
    subset = SHAP_DF[SHAP_DF["xvar_subset"] == feature_subset]
    models = subset["model_name"].unique()

    fig, axes = plt.subplots(1, len(models),figsize=(3 * len(models) + 1, 4),sharey=True)                              # +1 to make room for the colorbar column
    axes = np.atleast_1d(axes)

    vmin = subset[metric].min()
    vmax = subset[metric].max()
    cmap = plt.get_cmap('viridis')

    for ax, model in zip(axes, models):
        model_data = subset[subset["model_name"] == model]
        table = model_data.pivot_table(index="target", columns="scenario", values=metric)

        im = ax.imshow(table, aspect="auto", vmin=vmin, vmax=vmax, cmap = cmap)
        for i in range(table.shape[0]):
            for j in range(table.shape[1]):
                val = table.iloc[i, j]
                if pd.notna(val):
                    text_colour = get_text_colour(val, vmin, vmax, cmap)
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=12, color = text_colour)

        ax.set_xticks(range(len(table.columns)))
        ax.set_xticklabels(table.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(table.index)))
        ax.set_yticklabels(table.index, fontsize=8)
        ax.set_title(model, fontsize=10)

    fig.suptitle(f"{metric} ({feature_subset})", fontsize=13)

                                                                                                # saving space on the right for the colorbar BEFORE tight_layout runs
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])

                                                                                                # adding a  colorbar axis in that reserved space
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])                                             # [left, bottom, width, height] in figure fraction
    fig.colorbar(im, cax=cbar_ax, label=metric)

    plt.savefig(
        f"{base_dir}\heatmap_{metric}_{feature_subset}.png",
        bbox_inches='tight'
    )
    plt.close(fig)

metrics_to_plot = [
    'train_time', 'prediction_time', 'NRMSE', 'r2_test', 'train_test_mse_ratio', 'train-test_r2_diff'
]
for metric in metrics_to_plot:
    for feature_subset in ['small', 'all']:
        plot_heatmap(metric, feature_subset)



#################################################
# avg_stat_vs_initprop 
#################################################

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict


def create_magpie_mean_initprop_montage(df, statistic="mse_test"):                                              # Please note that create_magpie_mean_initprop_montage was created using Claude
    # -----------------------------
    # CONFIG
    # -----------------------------
    output_dir = rf"C:\Users\Sophie\OneDrive - National University of Ireland, Galway\Y1S2\Thesis\SHAP + RFR or PySR or NN\capstone2025-shiels-TIMES\{run_date}\Magpie_mean_results"
    scenarios = ['BASE_SSP2''1p5c_OS_SSP2','2C_SSP2','2C_SSP2_DA30']
    targets = ['GCost','GSupply','CO2eq','GConsumption']
    subsets = ['small','all']
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    # -----------------------------
    # FILTER MAGPIE
    # -----------------------------
    magpie_df = df[df["model_name"] == "Magpie"].copy()
    # Get init_prop values
    init_props = sorted(magpie_df["param2"].dropna().unique() )
    # Store generated plots
    plot_paths = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

    # -----------------------------
    # CREATE MEAN PLOTS
    # -----------------------------
    plot_dir = output_dir / "mean_plots"
    plot_dir.mkdir(
        exist_ok=True
    )
    for subset in subsets:
        for scenario in scenarios:
            for target in targets:
                filtered = magpie_df[
                    (magpie_df["xvar_subset"] == subset) &
                    (magpie_df["scenario"] == scenario) &
                    (magpie_df["target"] == target)
                ]
                if filtered.empty:
                    continue
                # mean statistic for each init_prop
                means = (filtered.groupby("param2")[statistic].mean().reset_index())plt.figure(figsize=(6,4))

                plt.plot(means["param2"],means[statistic],marker="o")
                plt.xlabel("Initial proportion (param2)")
                plt.ylabel(f"Mean {statistic}")
                plt.title(f"{scenario} | {target}")
                plt.grid(True)
                filename = (f"{subset}_{scenario}_{target}_mean_{statistic}.png")
                path = plot_dir / filename
                plt.tight_layout()
                plt.savefig(path,dpi=150)
                plt.close()
                plot_paths[subset][scenario][target] = path
    # -----------------------------
    # CREATE MONTAGES
    # -----------------------------
    for subset in subsets:
        rows = len(scenarios)
        cols = len(targets)
        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(
                cols*5,
                rows*4
            )
        )
        for row, scenario in enumerate(scenarios):
            for col, target in enumerate(targets):
                ax = axes[row][col]
                path = (plot_paths[subset].get(scenario,{}).get(target))
                if path:
                    img = Image.open(path)
                    ax.imshow(img)
                else:
                    ax.text(0.5, 0.5,"Missing",ha="center",va="center")
                ax.axis("off")
                # column headings
                if row == 0:
                    ax.set_title(target,fontsize=12,fontweight="bold")

            axes[row][0].annotate(
                scenario,
                xy=(-0.25,0.5),
                xycoords="axes fraction",
                rotation=90,
                fontsize=12,
                fontweight="bold",
                ha="center",
                va="center"
            )

            if row < rows-1:
                for col in range(cols):
                    axes[row][col].axhline(y=-0.05,color="black",linewidth=4,clip_on=False)

        plt.suptitle(
            f"Magpie Mean {statistic} vs Init Proportion - {subset}",
            fontsize=16,fontweight="bold", y=1.0)
        
        plt.tight_layout(rect=[0, 0, 1, 0.985])
        plt.subplots_adjust(left=0.12,top=0.92)

        montage_path = (output_dir /f"Magpie_mean_{statistic}_montage_{subset}.png")
        plt.savefig(montage_path,dpi=150,bbox_inches="tight")
        plt.close()
        print(
            f"Saved: {montage_path}"
        )
    print("Finished mean init_prop montages")

create_magpie_mean_initprop_montage(
    param_sweep_df,
    statistic="NRMSE"
)
#################################################
# magpie_vs_pysr 
#################################################

# metrics where higher = better; everything else assumed lower = better
HIGHER_IS_BETTER = {'r2_test', 'r2_train'}

def magpie_vs_pysr(stat):
    ascending = stat not in HIGHER_IS_BETTER
    n_rows = len(scenarios)
    n_cols = len(targets)

    for xvar_subset_code in xvar_subsets:
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(5 * n_cols, 4 * n_rows)
        )
        axes = np.atleast_2d(axes)

        for row, scenario in enumerate(scenarios):
            for col, target in enumerate(targets):
                ax = axes[row, col]

                magpie_data = param_sweep_df[
                    (param_sweep_df['model_name'] == 'Magpie') &
                    (param_sweep_df['xvar_subset'] == xvar_subset_code) &
                    (param_sweep_df['scenario'] == scenario) &
                    (param_sweep_df['target'] == target)
                ].copy()

                pysr_data = param_sweep_df[
                    (param_sweep_df['model_name'] == 'PySR') &
                    (param_sweep_df['xvar_subset'] == xvar_subset_code) &
                    (param_sweep_df['scenario'] == scenario) &
                    (param_sweep_df['target'] == target)
                ]
                pysr_stat = pysr_data[stat].mean()
                pysr_error = pysr_data[stat].std()
                pysr_error = 0 if pd.isna(pysr_error) else pysr_error

                group_counts = magpie_data.groupby(['param1', 'param2'])[stat].size()

                magpie_grouped = (
                    magpie_data.groupby(['param1', 'param2'])[stat]
                    .mean()
                    .reset_index()
                    .sort_values(['param2'], ascending=True)
                    .head(5)
                )

                errors = (
                    magpie_data.groupby(['param1', 'param2'])[stat]
                    .std()
                    .reindex(pd.MultiIndex.from_frame(magpie_grouped[['param1', 'param2']]))
                    .fillna(0)  # single-run configs have no std; show as 0 rather than dropping the bar
                    .values
                )

                magpie_labels = [f"max={int(r['param1'])}, init={r['param2']:g}"for _, r in magpie_grouped.iterrows()]
                all_labels = magpie_labels + ['PySR (mean)']

                ax.barh(
                    range(len(magpie_grouped)),
                    magpie_grouped[stat],
                    xerr=errors,
                    color='steelblue',
                    capsize=3,
                    label='Magpie'
                )
                ax.barh(
                    len(magpie_grouped),
                    pysr_stat,
                    xerr=pysr_error,
                    color='tomato',
                    capsize=3,
                    label='PySR'
                )

                ax.set_yticks(range(len(all_labels)))
                ax.set_yticklabels(all_labels, fontsize=7)
                ax.invert_yaxis()

                if row == 0:
                    ax.set_title(target, fontsize=10)
                if col == 0:
                    ax.set_ylabel(scenario, fontsize=9)
                if row == n_rows - 1:
                    ax.set_xlabel(stat) 

        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', ncol=2)

        fig.suptitle(f"Magpie vs PySR by {stat} — {xvar_subset_code}", fontsize=13)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(f"{param_sweep_dir}/top_parameters_by_{stat}_{xvar_subset_code}.pdf")
        plt.savefig(f"{param_sweep_dir}/top_parameters_by_{stat}_{xvar_subset_code}.png")
        plt.close(fig)
        print(f"Saved top parameters grid for {xvar_subset_code}")
        
magpie_vs_pysr('NRMSE')
magpie_vs_pysr('r2_test')
magpie_vs_pysr('train_test_mse_ratio')
magpie_vs_pysr('train-test_r2_diff')


## Statistical significance testing

The below code outlines a One-way ANOVA on NRMSE across 4 models (Magpie, PySR, RFR, MLP), balanced at n=10
per model per data subset (scenario/target combo).

Workflow:
  1. Loaded results.
  2. Per subset: Levene's test to test for equal variance -> one-way ANOVA -> Tukey HSD.
  3. Also ran Welch's ANOVA as a robustness check in case Levene's fails.
  4. Saved a summary table (F, p, eta-squared, Levene's p) across all subsets.

In [ ]:
# Please note that during the generation of this code, Claude was used to troubleshoot
# syntax issues

from scipy import stats
import pingouin as pg  # for Welch's ANOVA + Games-Howell;


# Path to results CSV (all 4 models, all runs, all subsets)
CSV_PATH = r"C:\Users\Sophie\OneDrive - National University of Ireland, Galway\Y1S2\Thesis\SHAP + RFR or PySR or NN\capstone2025-shiels-TIMES\20260823\Shap_outputs\model_comparisson_results1 - Copy.csv"


# Column names 
COL_NRMSE = "NRMSE"                                                                 # metric being investigated
COL_SUBSET = "xvar_subset"   
COL_MODEL = "model_name"         
COL_SCENARIO = "scenario"    
COL_TARGET = "target"       

df = pd.read_csv(CSV_PATH)
print(df.isna().sum())
print(df.groupby(COL_MODEL).size())                                                 # should be equal counts across all 4 models

MODELS_TO_SUBSAMPLE = ["Magpie", "PySRRegressor", "RandomForestRegressor", "MLPRegressor"]
N_TARGET = 30                                                                       # target number of runs per model per subset
RANDOM_STATE = 42                                                                   # for reproducible subsampling

OUTPUT_SUMMARY_CSV = "anova_summary_by_subset.csv"



def build_subset_key(df):
                                                                                    # Combining scenario/target/(feature subset) into one grouping key per combo
    keys = [df[COL_SCENARIO].astype(str), df[COL_TARGET].astype(str)]               # as it will be useful for printing names later
    if COL_SUBSET is not None and COL_SUBSET in df.columns:
        keys.append(df[COL_SUBSET].astype(str))
    return keys[0].str.cat(keys[1:], sep="_")


def run_anova_per_subset(df):
                                                                                        # Here we are running Levene's test, one-way ANOVA, Welch's ANOVA, and 
                                                                                        # Tukey HSD for each combo.
    results = []
    posthoc_tables = {}

    for combo_key, combo_df in df.groupby("combo_key"):
        groups = [g[COL_NRMSE].values for _, g in combo_df.groupby(COL_MODEL)]
        model_names = list(combo_df.groupby(COL_MODEL).groups.keys())

                                                                                        # Skipping if any group has < 2 observations as we can't estimate variance
        if any(len(g) < 2 for g in groups):
            print(f"Skipping {combo_key}: insufficient runs in at least one group.")
            continue

        # 1. Levene's test for equal variances
        levene_stat, levene_p = stats.levene(*groups)

        # 2. Standard one-way ANOVA 
        f_stat, anova_p = stats.f_oneway(*groups)

        # 3. Eta-squared effect size (SS_between / SS_total)
        all_vals = np.concatenate(groups)
        grand_mean = all_vals.mean()
        ss_total = np.sum((all_vals - grand_mean) ** 2)
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        eta_sq = ss_between / ss_total if ss_total > 0 else np.nan

        # 4. Welch's ANOVA via pingouin
        welch = pg.welch_anova(dv=COL_NRMSE, between=COL_MODEL, data=combo_df)
        welch_p = welch["p_unc"].iloc[0] if "p_unc" in welch.columns else welch["p-unc"].iloc[0]

        results.append({
            "combo": combo_key,
            "n_per_group": len(groups[0]),
            "levene_stat": levene_stat,
            "levene_p": levene_p,
            "equal_variance_ok": levene_p > 0.05,
            "anova_F": f_stat,
            "anova_p": anova_p,
            "eta_squared": eta_sq,
            "welch_p": welch_p,
            "significant_anova (p<0.05)": anova_p < 0.05,
            "significant_welch (p<0.05)": welch_p < 0.05,
        })

        # 5. Post-hoc: Tukey HSD (for equal-variance cases) and Games-Howell ( for unequal-variance cases)
        tukey = pg.pairwise_tukey(dv=COL_NRMSE, between=COL_MODEL, data=combo_df)
        games_howell = pg.pairwise_gameshowell(dv=COL_NRMSE, between=COL_MODEL, data=combo_df)
        posthoc_tables[combo_key] = {"tukey": tukey, "games_howell": games_howell}

    return pd.DataFrame(results), posthoc_tables


def main():
    df = pd.read_csv(CSV_PATH)
    df["combo_key"] = build_subset_key(df)

    print(f"Loaded {len(df)} rows across {df['combo_key'].nunique()} scenario/target combos.")
    print("Run counts per model before subsampling:")
    print(df.groupby(COL_MODEL).size())

    df_balanced = df

    print("\nRun counts per model after subsampling (should be <= 10 for Magpie/PySR):")
    print(df_balanced.groupby(COL_MODEL).size())

    summary, posthoc = run_anova_per_subset(df_balanced)

    print("\n=== ANOVA summary across all combos ===")
    print(summary.to_string(index=False))

    summary.to_csv(OUTPUT_SUMMARY_CSV, index=False)
    print(f"\nSaved summary to {OUTPUT_SUMMARY_CSV}")

                                                                                                    # Printing post-hoc tables for any significant combos
    sig_combos = summary.loc[summary["significant_anova (p<0.05)"], "combo"]
    if len(sig_combos) > 0:
        print("\n=== Post-hoc results (Tukey HSD) for significant combos ===")
        for combo in sig_combos:
            print(f"\n-- {combo} --")
            print(posthoc[combo]["tukey"].to_string(index=False))
    else:
        print("\nNo combos reached significance (p<0.05) in the standard ANOVA.")


if __name__ == "__main__":
    main()